In [ ]:
# 1. Osnovne biblioteke
import pandas as pd
import numpy as np

# 2. Vizualizacija
import matplotlib.pyplot as plt
import seaborn as sns

# 3. Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# 4. Modeli
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 5. Evaluacija
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv('cybersecurity_attacks.csv')
df

In [ ]:
stupci_za_izbacivanje = [
    'Timestamp', 'Source IP Address', 'Destination IP Address',
    'Payload Data', 'User Information', 'Device Information',
    'Proxy Information', 'Geo-location Data'
]

df = df.drop(columns=stupci_za_izbacivanje, errors='ignore')

df

In [ ]:
df.dtypes

In [ ]:
df[['Source Port', 'Destination Port',
    'Packet Length', 'Anomaly Scores']].describe()

In [ ]:
# Provjeravamo ima li duplih redaka
broj_duplikata = df.duplicated().sum()
print(f"Pronađeno duplikata: {broj_duplikata}")

In [ ]:
# Ispisuje koliko praznih polja ima u svakom stupcu
print(df.isnull().sum())

In [ ]:
df['Malware Indicators'] = df['Malware Indicators'].fillna('None')
df['Alerts/Warnings'] = df['Alerts/Warnings'].fillna('No Alert')
df['Firewall Logs'] = df['Firewall Logs'].fillna('No Log')
df['IDS/IPS Alerts'] = df['IDS/IPS Alerts'].fillna('No Alert')

In [ ]:
# Provjeri jedinstvene vrijednosti nakon fillna
print(df['Malware Indicators'].unique())
print(df['Alerts/Warnings'].unique())
print(df['Firewall Logs'].unique())
print(df['IDS/IPS Alerts'].unique())

In [ ]:
# Distribucija Attack Type, koliko ima svake vrste napada
df['Attack Type'].value_counts()

In [ ]:
plt.figure(figsize=(8,5))
df['Attack Type'].value_counts().plot(kind='bar', color=['#e74c3c','#3498db','#2ecc71'])
plt.title('Distribucija vrsta napada')
plt.xlabel('Vrsta napada')
plt.ylabel('Broj primjera')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# TARGET (ono što model predviđa)
y = df['Attack Type']

# Label encoding SAMO za target
le = LabelEncoder()
y = le.fit_transform(y)

# Provjeri mapiranje
print("Mapiranje klasa:")
for i, klasa in enumerate(le.classes_):
    print(f"  {i} → {klasa}")

# FEATURES (ulazni podaci)
X = df.drop(columns=['Attack Type'])

print(f"\nBroj featuresa: {X.shape[1]}")
print(f"Broj primjera:  {X.shape[0]}")

In [ ]:
# Dijelimo na 80% za učenje i 20% za testiranje
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

In [ ]:
# Pretvaramo tekstualne stupce u brojeve
X_train = pd.get_dummies(X_train, drop_first=True)
X_test  = pd.get_dummies(X_test,  drop_first=True)

# Osiguravamo iste stupce u train i test skupu
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Stupci se podudaraju: {list(X_train.columns) == list(X_test.columns)}")

In [ ]:
#DecisionTree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

In [ ]:
#RandomForest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
#LogisticRegression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=10000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

In [ ]:
#Evaluacija sva 3 modela
print("=" * 40)
print("DECISION TREE")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt,
      target_names=le.classes_))

print("=" * 40)
print("RANDOM FOREST")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf,
      target_names=le.classes_))

print("=" * 40)
print("LOGISTIC REGRESSION")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr,
      target_names=le.classes_))

In [ ]:
rezultati = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Logistic Regression'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_lr)
    ]
})

rezultati['Accuracy'] = rezultati['Accuracy'].round(4)
rezultati = rezultati.sort_values('Accuracy', ascending=False)
print(rezultati)

In [ ]:
# Confusion Matrix za sva 3 modela
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
modeli = [('Decision Tree', y_pred_dt), ('Random Forest', y_pred_rf), ('Logistic Regression', y_pred_lr)]

for ax, (naziv, y_pred) in zip(axes, modeli):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_,
                cmap='Blues')
    ax.set_title(naziv)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance - Random Forest
feat_imp = pd.Series(rf_model.feature_importances_,
                     index=X_train.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='bar', color='#3498db')
plt.title('Top 10 važnih obilježja - Random Forest')
plt.ylabel('Važnost')
plt.xlabel('Obilježje')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()